# GARCH-in-Mean: Premio de Risco e Volatilidade

Neste notebook, exploraremos o modelo **GARCH-M** (GARCH-in-Mean),
proposto por **Engle, Lilien & Robins (1987)**.

O GARCH-M permite que a **volatilidade condicional** entre diretamente na
equacao da media, capturando o **premio de risco** — a compensacao adicional
que investidores exigem por assumir maior risco.

**Conteudo:**
1. Relacao risco-retorno
2. O modelo GARCH-M
3. Variantes do GARCH-M
4. Interpretacao do parametro $\lambda$
5. EGARCH-M
6. Exercicios

**Referencias:**
- Engle, R.F., Lilien, D.M. & Robins, R.P. (1987). *Estimating time varying risk premia in the term structure: The ARCH-M model*. Econometrica, 55(2), 391-407.
- Bollerslev, T., Engle, R.F. & Wooldridge, J.M. (1988). *A capital asset pricing model with time-varying covariances*. Journal of Political Economy, 96(1), 116-131.
- Nelson, D.B. (1991). *Conditional heteroskedasticity in asset returns: A new approach*. Econometrica, 59(2), 347-370.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Relacao risco-retorno

A teoria financeira classica (CAPM, ICAPM) preve que ativos mais arriscados devem
oferecer retornos esperados maiores. Formalmente:

$$E[r_t | \mathcal{F}_{t-1}] = \mu + \lambda \cdot \text{Risco}_t$$

onde $\lambda > 0$ e o **preco do risco** (risk premium per unit of risk).

O **ICAPM** de Merton (1973) sugere que o retorno esperado de um ativo depende
da sua **variancia condicional**:

$$E[r_t | \mathcal{F}_{t-1}] = \mu + \lambda \cdot \sigma_t^2$$

Vamos verificar empiricamente se existe relacao positiva entre volatilidade e retorno.

In [ ]:
# TODO: Plote scatter retorno vs volatilidade realizada
# Dicas:
# - Carregue os dados: data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
# - returns = data['returns']
# - Calcule volatilidade realizada rolling: vol_20 = returns.rolling(20).std()
# - Calcule retorno medio rolling: ret_20 = returns.rolling(20).mean()
# - plt.scatter(vol_20, ret_20, alpha=0.3, s=10)
# - Adicione linha de regressao com np.polyfit
# - Observe: a relacao pode ser fraca ou ate negativa em janelas curtas

## 2. O modelo GARCH-M

O **GARCH-M** modela simultaneamente a media e a variancia condicional:

**Equacao da media:**
$$r_t = \mu + \lambda \cdot f(\sigma_t^2) + \epsilon_t$$

**Equacao da variancia:**
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

onde $\epsilon_t = \sigma_t z_t$ com $z_t \sim N(0,1)$.

O parametro $\lambda$ captura o **premio de risco**: se $\lambda > 0$,
periodos de alta volatilidade estao associados a maiores retornos esperados.

A funcao $f(\sigma_t^2)$ pode assumir diferentes formas:
- $f(\sigma_t^2) = \sigma_t$ (desvio padrao)
- $f(\sigma_t^2) = \sigma_t^2$ (variancia)
- $f(\sigma_t^2) = \log(\sigma_t^2)$ (log-variancia)

In [ ]:
# TODO: Estime GARCH-M com f(sigma)=sigma (desvio padrao)
# Dicas:
# - Carregue os dados se ainda nao carregou
# - data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
# - returns = data['returns']
# - model_vol = GARCHM(returns.values, p=1, q=1, risk_premium='volatility')
# - results_vol = model_vol.fit()
# - print(results_vol.summary())
# - Observe o valor de lambda e seu t-statistic

## 3. Variantes do GARCH-M

As tres formas mais comuns para $f(\sigma_t^2)$ sao:

| Variante | $f(\sigma_t^2)$ | Parametro archbox | Interpretacao de $\lambda$ |
|:---:|:---:|:---:|:---:|
| GARCH-M (vol) | $\sigma_t$ | `risk_premium='volatility'` | Premio por unidade de desvio padrao |
| GARCH-M (var) | $\sigma_t^2$ | `risk_premium='variance'` | Premio por unidade de variancia |
| GARCH-M (log) | $\log(\sigma_t^2)$ | `risk_premium='log_variance'` | Semi-elasticidade |

A escolha da forma funcional afeta a **interpretacao** do premio de risco,
mas nao a **estrutura** do modelo. Na pratica, a forma com $\sigma_t$
e mais intuitiva pois $\lambda$ tem a mesma unidade que o retorno.

In [ ]:
# TODO: Estime as 3 variantes e compare
# Dicas:
# - Variante 1 (volatility): model_vol = GARCHM(returns.values, risk_premium='volatility')
# - Variante 2 (variance): model_var = GARCHM(returns.values, risk_premium='variance')
# - Variante 3 (log_variance): model_log = GARCHM(returns.values, risk_premium='log_variance')
# - Ajuste os tres modelos e compare:
#   - Valor de lambda em cada variante
#   - AIC de cada variante: results.aic
#   - Persistencia (alpha+beta) de cada variante
# - Faca uma tabela comparativa

## 4. Interpretacao do parametro $\lambda$

O parametro $\lambda$ e central no GARCH-M:

- **$\lambda > 0$**: premio de risco positivo — investidores exigem maior retorno
  esperado em periodos de alta volatilidade (consistente com teoria)
- **$\lambda = 0$**: sem premio de risco — volatilidade nao afeta o retorno esperado
  (o GARCH-M reduz ao GARCH padrao)
- **$\lambda < 0$**: premio de risco negativo — retornos menores em periodos de alta
  volatilidade (pode ocorrer em periodos de panico)

Para testar se $\lambda$ e estatisticamente significativo:

$$H_0: \lambda = 0 \quad \text{vs} \quad H_1: \lambda \neq 0$$

Usamos a estatistica $t = \hat{\lambda} / \text{se}(\hat{\lambda})$.

In [ ]:
# TODO: Teste significancia de lambda
# Dicas:
# - Use os resultados da variante com volatility: results_vol
# - O lambda esta em results_vol.params[-1] (ultimo parametro)
# - O t-value esta em results_vol.tvalues[-1]
# - O p-value esta em results_vol.pvalues[-1]
# - Imprima: lambda, t-stat, p-valor
# - Se p-valor < 0.05, rejeita H0 (lambda e significativo)
# - Visualize o premio de risco ao longo do tempo:
#   risk_premium = results_vol.params[-1] * results_vol.conditional_volatility
#   Use plot_garchm_risk_premium() para visualizar

## 5. EGARCH-M

Podemos combinar o efeito de **assimetria** (leverage) com o **premio de risco**
usando um modelo EGARCH na equacao da variancia:

**Equacao da media:**
$$r_t = \mu + \lambda \cdot \sigma_t + \epsilon_t$$

**Equacao da variancia (EGARCH):**
$$\log(\sigma_t^2) = \omega + \alpha |z_{t-1}| + \gamma z_{t-1} + \beta \log(\sigma_{t-1}^2)$$

O parametro $\gamma < 0$ captura o **efeito alavancagem**: choques negativos
($z_{t-1} < 0$) aumentam mais a volatilidade do que choques positivos.

O EGARCH-M combina dois fatos estilizados:
1. Volatilidade assimetrica (efeito alavancagem)
2. Premio de risco variante no tempo

In [ ]:
# TODO: Estime EGARCH-M
# Dicas:
# - Primeiro estime um EGARCH puro para referencia:
#   model_egarch = EGARCH(returns.values, p=1, q=1)
#   results_egarch = model_egarch.fit()
# - O archbox permite combinar EGARCH com risk_premium via GARCHM
#   ou voce pode construir o premio manualmente:
#   premium = results_egarch.conditional_volatility * lambda_hat
# - Compare o AIC do EGARCH com o do GARCH-M
# - Observe se o efeito alavancagem (gamma) e significativo

## 6. Exercicios

1. **GARCH vs GARCH-M**: Estime um GARCH(1,1) padrao e compare com o GARCH-M.
   O premio de risco melhora o ajuste do modelo (menor AIC)?

2. **Distribuicao t-Student**: Re-estime o GARCH-M com distribuicao t-Student
   para os erros. O lambda muda significativamente?

3. **Subamostras**: Divida os dados em duas metades e estime o GARCH-M em cada
   uma. O lambda e estavel ao longo do tempo?

4. **Impacto economico**: Calcule a diferenca no retorno esperado entre um
   periodo de baixa volatilidade (percentil 10) e alta volatilidade (percentil 90).
   O premio de risco e economicamente significativo?

In [ ]:
# TODO: Compare GARCH(1,1) vs GARCH-M vs EGARCH-M
# Dicas:
# - Estime GARCH(1,1): model_g = GARCH(returns.values, p=1, q=1); res_g = model_g.fit()
# - Ja temos results_vol (GARCH-M com volatility)
# - Ja temos results_egarch (EGARCH)
# - Compare AIC e BIC dos tres modelos:
#   print(f'GARCH(1,1): AIC={res_g.aic:.2f}, BIC={res_g.bic:.2f}')
#   print(f'GARCH-M:    AIC={results_vol.aic:.2f}, BIC={results_vol.bic:.2f}')
#   print(f'EGARCH:     AIC={results_egarch.aic:.2f}, BIC={results_egarch.bic:.2f}')
# - Plote as volatilidades condicionais dos 3 modelos
# - Qual modelo e preferido pelos criterios de informacao?